# Testing WAR tables with Lahman DB

Load the BBRef WAR CSVs into `lahman.db` as new tables, then join with existing Lahman data.

In [5]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("lahman.db")
conn.execute("PRAGMA foreign_keys = ON;")

# Load WAR CSVs
batting_war = pd.read_csv("BBRef_Batting_WAR.csv")
pitching_war = pd.read_csv("BBRef_Pitching_WAR.csv")

print("Batting WAR sample:")
print(batting_war.head())
print(f"\nPitching WAR sample:")
print(pitching_war.head())

Batting WAR sample:
   yearID    bbrefID            Player Team  WAR
0    2024  duranja01      Jarren Duran  BOS  9.0
1    2024  ohtansh01     Shohei Ohtani  LAD  9.0
2    2024  hendegu01  Gunnar Henderson  BAL  8.8
3    2024  semiema01     Marcus Semien  TEX  4.5
4    2024   sotoju01         Juan Soto  NYY  8.0

Pitching WAR sample:
   yearID    bbrefID         Player Team  WAR
0    2024  gilbelo01  Logan Gilbert  SEA  3.0
1    2024   lugose01      Seth Lugo  KCR  5.0
2    2024   webblo01     Logan Webb  SFG  3.8
3    2024  wheelza01   Zack Wheeler  PHI  6.5
4    2024   nolaaa01     Aaron Nola  PHI  4.0


In [6]:
# Add WAR tables to the database
batting_war.to_sql("BattingWAR", conn, if_exists="replace", index=False)
pitching_war.to_sql("PitchingWAR", conn, if_exists="replace", index=False)

print(f"BattingWAR: {len(batting_war)} rows loaded")
print(f"PitchingWAR: {len(pitching_war)} rows loaded")

BattingWAR: 1817 rows loaded
PitchingWAR: 2245 rows loaded


## Aaron Judge — HRs and WAR (2024–2025)

Join path: `Batting` → `People` (on playerID) → `BattingWAR` (on bbrefID + yearID) → `Teams` (on yearID + teamID, using teamIDBR to match BBRef's team codes)

In [7]:
pd.read_sql_query("""
    SELECT b.yearID,
           p.nameFirst || ' ' || p.nameLast AS name,
           b.teamID,
           b.HR,
           w.WAR
    FROM Batting b
    JOIN People p ON b.playerID = p.playerID
    JOIN Teams t ON b.teamID = t.teamID AND b.yearID = t.yearID
    JOIN BattingWAR w ON p.bbrefID = w.bbrefID
                     AND b.yearID = w.yearID
                     AND t.teamIDBR = w.Team
    WHERE p.nameLast = 'Judge' AND p.nameFirst = 'Aaron'
      AND b.yearID BETWEEN 2024 AND 2025
    ORDER BY b.yearID
""", conn)

,yearID,name,teamID,HR,WAR
0,2024,Aaron Judge,NYA,58,10.9
1,2025,Aaron Judge,NYA,53,9.7


In [8]:
# Aaron Judge in BattingWAR table
pd.read_sql_query("""
    SELECT * FROM BattingWAR
    WHERE Player LIKE '%Judge%'
""", conn)

,yearID,bbrefID,Player,Team,WAR
0,2024,judgeaa01,Aaron Judge,NYY,10.9
1,2025,judgeaa01,Aaron Judge,NYY,9.7


In [ ]:
conn.close()